# Radial Basis Function

The scope of this exercise is to predict whether a bank customer will be eligible for a term deposit. The prediction will be based on selected demographic and personal characteristics, such as age, occupation, education level, marital status, and other relevant features.

To perform this task, we will use a neural-network approach based on Radial Basis Functions (RBF). In this model, the centers of the hidden-layer neurons are determined using the k-means clustering algorithm.




Step 1: Loading the required libraries.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import math
from sklearn.metrics import accuracy_score

**Step 2**: Loadinf of the bank-full.csv dataset and keeping only the columns that will be used as input features, such as the customer’s job, loan status, and other relevant attributes. The dataset is available [here](https://raw.githubusercontent.com/netmode/Stochastic-Processes-and-Optimization-in-Machine-Learning-Lab/refs/heads/main/lab9/bank-full.csv)
. Download and upload to your Colab environment before running the code.

In [ ]:
dataset = pd.read_csv('bank-full.csv',sep=None)

In [ ]:
dataset.head()

In [ ]:
col_to_use = ['age','balance','day','duration','campaign','pdays','previous']
data = dataset.drop(col_to_use,axis=1)

In [ ]:
data.head()

In [ ]:
data = data.apply(LabelEncoder().fit_transform)

In [ ]:
data.head()

In [ ]:
data['job'].unique()

In [ ]:
dataset['job'].unique()

In [ ]:
data_rest = dataset[col_to_use]
data_rest.head()

In [ ]:
dataset2 = pd.concat([data_rest,data],axis=1)

In [ ]:
dataset2.head()

**Step 3**: Splitinf of the dataset into training and test sets, and scaling of the features using the StandardScaler() function.

In [ ]:
X= dataset2.drop('y',axis=1)
y= dataset2['y']

X_train, X_test, y_train, y_test= train_test_split(X,y, test_size= 0.33, random_state= 4)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
X_train

**Step 4**: Determining the centers of the hidden-layer neurons using the k-means algorithm. First, we apply k-means to the training data to find the cluster centers, and then compute the standard deviation of the clusters.

In [ ]:
K_cent= 8
km= KMeans(n_clusters= K_cent, max_iter= 100)
km.fit(X_train)
cent= km.cluster_centers_

In [ ]:
max=0
for i in range(K_cent):
	for j in range(K_cent):
		d= np.linalg.norm(cent[i]-cent[j])
		if(d> max):
			max= d
d= max

sigma= d/math.sqrt(2*K_cent)

**Step 5**: Construction of the F matrix, where each row corresponds to an input sample elements and each column represents the output of one of the K radial basis functions φj(x).

In [ ]:
shape= X_train.shape
row= shape[0]
column= K_cent
F= np.empty((row,column), dtype= float)

In [ ]:
for i in range(row):
  for j in range(column):
    dist= np.linalg.norm(X_train[i]-cent[j])
    F[i][j]= math.exp(-math.pow(dist,2)/math.pow(2*sigma,2))

**Step 6**: Calculation of the weight matrix W.

In [ ]:
FTG= np.dot(F.T,F)
FTG_inv= np.linalg.inv(FTG)
fac= np.dot(FTG_inv,F.T)
W= np.dot(fac,y_train)

**Step 7**: Building of the F matrix using the test dataset.

In [ ]:
row= X_test.shape[0]
column= K_cent
F_test= np.empty((row,column), dtype= float)
for i in range(row):
	for j in range(column):
		dist= np.linalg.norm(X_test[i]-cent[j])
		F_test[i][j]= math.exp(-math.pow(dist,2)/math.pow(2*sigma,2))

**Step 8**: Evaluation of the prediction accuracy on the test dataset.

In [ ]:
prediction= np.dot(F_test,W)
prediction= 0.5*(np.sign(prediction-0.5)+1)

score= accuracy_score(prediction,y_test)
print(score)

## Questions

**Question 1:** Briefly describe how a Radial Basis Function (RBF) neural network works.

**Question 2:** What methods can be used to determine the weights in an RBF network? Explain the main differences between these methods.

**Question 3:** What is the role of the σ (sigma) parameter in an RBF network? Explain how it affects the model’s performance. What may happen if σ is set to a very small or very large value?